# 可选实验：Feature Engineering 与 Polynomial Regression

![Feature Engineering 课程示意图](./images/C1_W2_Lab07_FeatureEngLecture.PNG)


## 目标

在本实验中，你将：

- 探索 feature engineering（特征工程）和 polynomial regression（多项式回归），学习如何使用 linear regression 的方法拟合复杂、甚至高度非线性的函数。


## 工具

你将使用之前实验中编写的函数，以及 Matplotlib 和 NumPy。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import zscore_normalize_features, run_gradient_descent_feng
np.set_printoptions(precision=2)  # reduced display precision on numpy arrays

<a name="FeatureEng"></a>
## Feature Engineering 与 Polynomial Regression 概述

Linear regression 可以直接构建如下形式的模型：

$$
f_{\mathbf{w},b}(\mathbf{x})=w_0x_0+w_1x_1+\cdots+w_{n-1}x_{n-1}+b \tag{1}
$$

如果 feature 与 target 的关系是非线性的，或者需要组合多个 feature，该怎么办？例如，房价与居住面积往往并非简单的线性关系：过小或过大的房屋可能使价格曲线偏离直线，如上图所示。

如何使用 linear regression 拟合这条曲线？我们目前掌握的方法，是调整公式 (1) 中的参数 $\mathbf{w}$ 和 $b$，使模型拟合训练数据。然而，在 feature 保持不变时，仅调整这些参数无法拟合非线性曲线。


<a name="PolynomialFeatures"></a>
### Polynomial Features

上面讨论了数据呈非线性关系的情况。下面尝试运用已有知识拟合一条非线性曲线，从简单的 quadratic（二次函数）$y=1+x^2$ 开始。

这里使用的函数都已在之前的实验中介绍，可以在 `lab_utils.py` 中查看。我们还会使用 NumPy 的 [`np.c_[..]`](https://numpy.org/doc/stable/reference/generated/numpy.c_.html)，将数组按列拼接。


In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = 1 + x**2
X = x.reshape(-1, 1)

model_w,model_b = run_gradient_descent_feng(X,y,iterations=1000, alpha = 1e-2)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("no feature engineering")
plt.plot(x,X@model_w + model_b, label="Predicted Value");  plt.xlabel("X"); plt.ylabel("y"); plt.legend(); plt.show()

结果与预期一样，拟合效果并不好。我们需要的是类似 $y=w_0x_0^2+b$ 的形式，也就是增加一个 **polynomial feature**。

为此，可以修改*输入数据*，构造所需的 feature。如果将原始 $x$ 值替换为其平方，就可以得到 $y=w_0x_0^2+b$。下面将 `X` 替换为 `X**2`，试试看。


In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = 1 + x**2

# Engineer features 
X = x**2      #<-- added engineered feature

In [ ]:
X = X.reshape(-1, 1)  #X should be a 2-D Matrix
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha = 1e-5)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Added x**2 feature")
plt.plot(x, np.dot(X,model_w) + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

现在的拟合效果接近完美！请观察图上方打印的 $\mathbf{w}$ 和 $b$：`w,b found by gradient descent: w: [1.], b: 0.0490`。

Gradient descent 将初始参数调整为约 `(1.0, 0.049)`，对应模型 $y=1\cdot x_0^2+0.049$，在图示尺度下已非常接近目标 $y=1\cdot x_0^2+1$。增加迭代次数还可以进一步改善拟合。


<a name="GDF"></a>
#### 选择 Features

上面的例子中，我们事先知道需要 $x^2$ 项，但实际问题中不一定能直接判断所需的 feature。可以加入多种候选 feature，再尝试找出最有用的部分。例如，如果尝试 $y=w_0x+w_1x^2+w_2x^3+b$，会发生什么？

请运行下面的代码单元。


In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = x**2

# engineer features .
X = np.c_[x, x**2, x**3]   #<-- added engineered feature

In [ ]:
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha=1e-7)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("x, x**2, x**3 features")
plt.plot(x, X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

注意，$\mathbf{w}$ 为 `[0.08 0.54 0.03]`，$b$ 为 `0.0106`，因此训练后的模型为：

$$
f_{\mathbf{w},b}(x)=0.08x+0.54x^2+0.03x^3+0.0106
$$

Gradient descent 使 $w_1$ 相对于其他 weight 更大，从而突出最符合目标数据的 $x^2$ feature。如果运行足够长时间，其他项的影响还会继续减小。

> 在这个示例中，gradient descent 通过增大对应的 weight，突出更适合拟合数据的 feature。

回顾这个例子的含义：

- 当某项 weight 变为 0 或非常接近 0 时，对应 feature 对当前模型预测的贡献也会变小。比较不同 feature 的 weight 大小时，还需要考虑它们的尺度。
- 在这里，拟合后 $x^2$ 对应的 weight 明显大于 $x$ 和 $x^3$ 对应的 weight，因为 $x^2$ 最适合拟合目标数据。


#### 换一个角度理解

前面根据 polynomial features 与目标数据的匹配程度来选择 feature。也可以这样理解：即使创建了新的 feature，我们使用的仍然是 linear regression。因此，理想的 feature 应当与 target 呈线性关系。下面通过示例说明。


In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = x**2

# engineer features .
X = np.c_[x, x**2, x**3]   #<-- added engineered feature
X_features = ['x','x^2','x^3']

In [ ]:
fig,ax=plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X[:,i],y)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("y")
plt.show()

从上图可以明显看出，$x^2$ feature 与 target $y$ 之间是线性关系。因此，linear regression 很容易利用该 feature 构建模型。


#### 对 Features 进行 Scaling

上一个实验介绍过，当不同 feature 的尺度差异较大时，应使用 feature scaling 来加快 gradient descent。在上面的示例中，$x$、$x^2$ 和 $x^3$ 的尺度自然会有很大差异。下面对它们应用 z-score normalization。


In [ ]:
# create target data
x = np.arange(0,20,1)
X = np.c_[x, x**2, x**3]
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X,axis=0)}")

# add mean_normalization 
X = zscore_normalize_features(X)     
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X,axis=0)}")

现在可以使用更大的 $\alpha$ 再次尝试：


In [ ]:
x = np.arange(0,20,1)
y = x**2

X = np.c_[x, x**2, x**3]
X = zscore_normalize_features(X) 

model_w, model_b = run_gradient_descent_feng(X, y, iterations=100000, alpha=1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

Feature scaling 显著加快了收敛速度。

再次观察 $\mathbf{w}$：对应 $x^2$ 的 $w_1$ 最为突出，gradient descent 几乎消除了 $x^3$ 项的影响。


#### 复杂函数

通过 feature engineering，甚至可以为相当复杂的函数建立模型：


In [ ]:
x = np.arange(0,20,1)
y = np.cos(x/2)

X = np.c_[x, x**2, x**3,x**4, x**5, x**6, x**7, x**8, x**9, x**10, x**11, x**12, x**13]
X = zscore_normalize_features(X) 

model_w,model_b = run_gradient_descent_feng(X, y, iterations=1000000, alpha = 1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()


## 恭喜完成实验！

在本实验中，你：

- 学习了如何通过 feature engineering，使用 linear regression 拟合复杂、甚至高度非线性的函数。
- 理解了在 feature engineering 中应用 feature scaling 的重要性。
